# CSP-basierte Verteilung zentraler Prüfungsräume

**Individueller Datensatz – Master KI (CSP)**

Dieses Notebook modelliert und löst die Verteilung von 40 zentralen Raum-Zeit-Slots pro Prüfungswoche als Constraint Satisfaction Problem (CSP). Es wird keine Zielfunktion optimiert; gesucht wird eine *zulässige* Belegung, die alle formulierten Constraints erfüllt.

**Struktur des Notebooks**
1. Problemanalyse und formale CSP-Modellierung
2. Ableitung der Bedarfs- und Fairness-Constraints
3. Implementierung des Constraint-Modells
4. Lösungen der Prüfungswochen und Validierung
5. Analyse des Lösungsraums
6. Experiment mit verändertem Constraint
7. Unlösbarkeit und Konfliktanalyse
8. Analyse des Suchverfahrens
9. Kritische Reflexion
10. Dokumentation der GenAI-Nutzung
11. Fazit

## 1. Problemanalyse und formale CSP-Modellierung

### 1.1 Entscheidungen des Planungssystems
Für jede Prüfungswoche muss entschieden werden, **welcher Studiengang** jeden der 40 Raum-Zeit-Slots (5 Tage × 4 Zeitslots × 2 Räume) belegt. Es ist stets genau ein Studiengang pro Slot zuzuweisen.

### 1.2 Relativer Bedarf aus Kursgruppen
Sei $g_s$ die Anzahl der Kursgruppen des Studiengangs $s$ in der betrachteten Woche und $W$ die Menge der beteiligten Studiengänge. Dann ist der relative Bedarf

$$q_s = \frac{g_s}{\sum_{j \in W} g_j}.$$

Der erwartete Slot-Anteil ergibt sich als $E_s = 40 \cdot q_s$. Da nur ganzzahlige Slots vergeben werden, werden nachvollziehbare Unter- und Obergrenzen $L_s = \lfloor E_s \rfloor - 1$ und $U_s = \lceil E_s \rceil + 1$ definiert.

### 1.3 Organisatorische Anforderungen als Constraints
- **Slot-Eindeutigkeit:** Jeder Raum-Zeit-Slot wird genau einem Studiengang zugewiesen.
- **Wochenbeschränkung:** Nur in der Woche beteiligte Studiengänge erhalten Slots.
- **Bedarfsgerechtigkeit:** $L_s \le \text{count}_s \le U_s$.
- **Tagesverteilung:** Kein Studiengang konzentriert seine Slots auf einen Tag.
- **Popularitäts-Fairness:** Mittlere Popularität pro Slot liegt nahe am globalen Mittel.
- **Parallel / Sequenziell:** Abhängig von der Kursgruppenzahl.
- **Kein später→früher-Übergang:** Wer am Vortag den letzten Slot hat, darf am Folgetag nicht den ersten Slot belegen.

### 1.4 Slot-lokale vs. koppelnde Constraints
- **Slot-lokal:** Slot-Eindeutigkeit, Wochenbeschränkung.
- **Koppelnd:** Bedarfsgrenzen, Tagesverteilung, Popularitäts-Fairness, Parallel/Sequenziell, Übergangsregel.

### 1.5 Formale CSP-Definition
$$CSP = (X, D, C)$$

- **Variablen** $X$: Für jede Woche, jeden Tag $d \in \{0,\dots,4\}$, jeden Zeitslot $t \in \{0,\dots,3\}$, jeden Raum $r \in \{0,1\}$ und jeden Studiengang $s$ eine Boolesche Variable $y_{s,d,t,r}$.
- **Domänen** $D$: $y_{s,d,t,r} \in \{0,1\}$.
- **Constraints** $C$: Siehe Abschnitt 2.

## 2. Ableitung der Bedarfs- und Fairness-Constraints

### 2.1 Bedarf
$L_s = \max(0, \lfloor E_s \rfloor - 1)$ und $U_s = \min(40, \lceil E_s \rceil + 1)$. Für die gegebene Datenlage ergeben sich exakte Werte, sodass die Toleranz $\pm 1$ ausreicht.

### 2.2 Tagesverteilung
Falls $E_s \ge 5$, mindestens ein Slot pro Tag; falls $E_s \ge 10$, mindestens zwei Slots pro Tag.

### 2.3 Popularität
Additive Kombination: $p(d,t) = \text{weekday\_pref}[d] + \text{time\_pref}[t]$.
Fairness: $\left| \frac{\sum p(d,t) \cdot y_{s,d,t,r}}{\text{count}_s} - \overline{p} \right| \le 2.0$, umgesetzt als lineare Ungleichung mit skalierten Ganzzahlen.

### 2.4 Parallel / Sequenziell
- $g_s = 1$: Keine parallele Nutzung beider Räume, keine direkt aufeinanderfolgenden Zeitslots.
- $g_s \ge 3$: Mindestens $\max(1, \lfloor g_s/4 \rfloor)$ parallele Belegungen.
- $g_s = 2$: Keine zusätzliche Einschränkung.

### 2.5 Übergangsregel
$y_{s,d,D,r_1} + y_{s,d+1,A,r_2} \le 1$ für alle $d \in \{0,\dots,3\}$, $r_1, r_2 \in \{0,1\}$.

In [ ]:
!pip install ortools

In [ ]:
# ============================================================
# 3. Implementierung des Constraint-Modells
# ============================================================
import csv, json, math
from ortools.sat.python import cp_model

# ---------- 3.1 Daten einlesen ----------
weeks = {}
with open('exam_weeks.csv', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter=';')
    for row in reader:
        w = row['exam_week'].strip()
        s = row['study_program'].strip()
        g = int(row['course_groups'])
        cohorts = int(row['cohorts'])
        # effektive Kursgruppen = Kursgruppen * Jahrgänge
        weeks.setdefault(w, {})[s] = g * cohorts

with open('preferences.json', encoding='utf-8') as f:
    prefs = json.load(f)

time_pref = {k: v['preference'] for k, v in prefs['time_slot_preferences'].items()}
day_pref  = prefs['weekday_preferences']
structure_rules = prefs['appointment_structure_preferences']

days  = ['Montag', 'Dienstag', 'Mittwoch', 'Donnerstag', 'Freitag']
times = ['A', 'B', 'C', 'D']
rooms = [0, 1]

def p(d, t):
    """Popularitätsmaß eines Slots (Tag d, Zeitslot t)."""
    return day_pref[days[d]] + time_pref[times[t]]

def global_avg_pop():
    total = sum(p(d, t) for d in range(5) for t in range(4) for r in range(2))
    return total / 40.0

print('Wochen:', {w: list(p.keys()) for w, p in weeks.items()})
print('Globaler Popularitäts-Mittelwert:', global_avg_pop())

Wochen: {'Woche 1': ['ES', 'INF', 'MB', 'MT', 'DS'], 'Woche 2': ['ET', 'MB', 'MT', 'DS'], 'Woche 3': ['WIW', 'ET', 'MB', 'DS']}
Globaler Popularitäts-Mittelwert: 6.65


In [ ]:
# ============================================================
# 3.2 CSP-Modell pro Woche
# ============================================================
POP_TOLERANCE = 2.0  # zulässige Abweichung der mittleren Popularität

def solve_week(week_name, programs, pop_tol=POP_TOLERANCE, time_limit=15.0,
               exclude=None, extra_constraints=None):
    """Baut und löst das CSP für eine Woche.
    exclude: dict {(s,d,t,r): 1} - erzwingt, dass die Variable NICHT gesetzt wird
    extra_constraints: Funktion(model, y, count) für zusätzliche Bedingungen
    """
    model = cp_model.CpModel()
    W = list(programs.keys())

    # Variablen
    y = {}
    for s in W:
        for d in range(5):
            for t in range(4):
                for r in rooms:
                    y[(s, d, t, r)] = model.NewBoolVar(f'y_{s}_{d}_{t}_{r}')

    # Slot-Eindeutigkeit
    for d in range(5):
        for t in range(4):
            for r in rooms:
                model.Add(sum(y[(s, d, t, r)] for s in W) == 1)

    # Slot-Zähler pro Studiengang
    count = {s: sum(y[(s, d, t, r)]
                    for d in range(5) for t in range(4) for r in rooms)
             for s in W}

    # Bedarfsgrenzen
    G = sum(programs.values())
    for s, g in programs.items():
        E = g / G * 40.0
        lower = max(0, math.floor(E) - 1)
        upper = min(40, math.ceil(E) + 1)
        model.Add(count[s] >= lower)
        model.Add(count[s] <= upper)

    # Tagesverteilung
    for s, g in programs.items():
        E = g / G * 40.0
        min_per_day = 0
        if E >= 5:
            min_per_day = 1
        if E >= 10:
            min_per_day = 2
        for d in range(5):
            model.Add(sum(y[(s, d, t, r)] for t in range(4) for r in rooms)
                      >= min_per_day)

    # Popularitäts-Fairness (skaliert mit 10 auf Ganzzahlen)
    avg_int = int(global_avg_pop() * 10)
    tol_int = int(pop_tol * 10)
    for s in W:
        sum_p = sum(y[(s, d, t, r)] * (p(d, t) * 10)
                    for d in range(5) for t in range(4) for r in rooms)
        model.Add(sum_p <= (avg_int + tol_int) * count[s])
        model.Add(sum_p >= (avg_int - tol_int) * count[s])

    # Parallel / Sequenziell
    for s, g in programs.items():
        if g == 1:
            # keine parallele Nutzung beider Räume
            for d in range(5):
                for t in range(4):
                    model.Add(y[(s, d, t, 0)] + y[(s, d, t, 1)] <= 1)
            # keine direkt aufeinanderfolgenden Zeitslots
            for d in range(5):
                for t in range(3):
                    model.Add(sum(y[(s, d, t, r)] for r in rooms)
                              + sum(y[(s, d, t + 1, r)] for r in rooms) <= 1)
        elif g >= 3:
            K = max(1, g // 4)
            par_vars = []
            for d in range(5):
                for t in range(4):
                    pv = model.NewBoolVar(f'par_{s}_{d}_{t}')
                    model.Add(pv <= y[(s, d, t, 0)])
                    model.Add(pv <= y[(s, d, t, 1)])
                    model.Add(pv >= y[(s, d, t, 0)] + y[(s, d, t, 1)] - 1)
                    par_vars.append(pv)
            model.Add(sum(par_vars) >= K)

    # Kein später->früher-Übergang
    for s in W:
        for d in range(4):
            for r1 in rooms:
                for r2 in rooms:
                    model.Add(y[(s, d, 3, r1)] + y[(s, d + 1, 0, r2)] <= 1)

    # optionale Ausschlüsse (für Lösungsraum-Analyse)
    if exclude:
        for (s, d, t, r), val in exclude.items():
            if val == 1:
                model.Add(y[(s, d, t, r)] == 0)

    # zusätzliche Constraints (für Experimente)
    if extra_constraints is not None:
        extra_constraints(model, y, count, programs)

    # Lösen
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    solver.parameters.random_seed = 0
    status = solver.Solve(model)

    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        schedule = {}
        for d in range(5):
            for t in range(4):
                for r in rooms:
                    for s in W:
                        if solver.Value(y[(s, d, t, r)]) == 1:
                            schedule[(d, t, r)] = s
        return schedule, solver, status
    return None, solver, status

In [ ]:
# ============================================================
# 3.3 Validierung: Prüft alle Constraints nach
# ============================================================
def validate(schedule, programs, pop_tol=POP_TOLERANCE, verbose=True):
    W = list(programs.keys())
    errors = []

    # 1) Jeder Slot genau einmal belegt
    for d in range(5):
        for t in range(4):
            for r in rooms:
                if (d, t, r) not in schedule:
                    errors.append(f'Slot ({d},{t},{r}) unbelegt')

    # 2) Nur beteiligte Studiengänge
    for k, s in schedule.items():
        if s not in W:
            errors.append(f'Nicht beteiligter Studiengang {s} in Slot {k}')

    # 3) Bedarfsgrenzen
    G = sum(programs.values())
    count = {s: 0 for s in W}
    for k, s in schedule.items():
        count[s] += 1
    for s, g in programs.items():
        E = g / G * 40.0
        lo = max(0, math.floor(E) - 1)
        hi = min(40, math.ceil(E) + 1)
        if not (lo <= count[s] <= hi):
            errors.append(f'{s}: count={count[s]} nicht in [{lo},{hi}]')

    # 4) Popularitäts-Fairness
    avg = global_avg_pop()
    for s in W:
        if count[s] == 0:
            continue
        sum_p = sum(p(d, t) for (d, t, r), ss in schedule.items() if ss == s)
        m = sum_p / count[s]
        if abs(m - avg) > pop_tol + 1e-6:
            errors.append(f'{s}: mittlere Popularität {m:.2f} weicht zu stark ab')

    # 5) Parallel / Sequenziell
    for s, g in programs.items():
        if g == 1:
            for d in range(5):
                for t in range(4):
                    if schedule.get((d, t, 0)) == s and schedule.get((d, t, 1)) == s:
                        errors.append(f'{s}: parallele Belegung trotz g=1 ({d},{t})')
                for t in range(3):
                    a = sum(1 for r in rooms if schedule.get((d, t, r)) == s)
                    b = sum(1 for r in rooms if schedule.get((d, t + 1, r)) == s)
                    if a > 0 and b > 0:
                        errors.append(f'{s}: aufeinanderfolgende Slots trotz g=1 ({d},{t})')
        elif g >= 3:
            K = max(1, g // 4)
            par = sum(1 for d in range(5) for t in range(4)
                      if schedule.get((d, t, 0)) == s and schedule.get((d, t, 1)) == s)
            if par < K:
                errors.append(f'{s}: nur {par} parallele Belegungen, gefordert {K}')

    # 6) Kein später->früher-Übergang
    for s in W:
        for d in range(4):
            late = any(schedule.get((d, 3, r)) == s for r in rooms)
            early = any(schedule.get((d + 1, 0, r)) == s for r in rooms)
            if late and early:
                errors.append(f'{s}: später Slot an Tag {d}, früher an Tag {d+1}')

    if verbose:
        if errors:
            print('Validierung FEHLGESCHLAGEN:')
            for e in errors:
                print(' -', e)
        else:
            print('Validierung erfolgreich – alle Constraints erfüllt.')
    return errors

## 4. Lösungen der Prüfungswochen und Validierung
Für jede Woche wird das CSP gelöst, der Belegungsplan ausgegeben und die Constraints überprüft.

In [ ]:
def print_schedule(schedule, programs):
    header = f"{'Tag':<12} {'Zeit':<6} {'Raum 0':<8} {'Raum 1':<8}"
    print(header)
    print('-' * len(header))
    for d in range(5):
        for t in range(4):
            s0 = schedule.get((d, t, 0), '-')
            s1 = schedule.get((d, t, 1), '-')
            print(f"{days[d]:<12} {times[t]:<6} {s0:<8} {s1:<8}")
    print('\nZugewiesene Slots pro Studiengang:')
    counts = {s: 0 for s in programs}
    for k, s in schedule.items():
        counts[s] += 1
    G = sum(programs.values())
    for s, g in programs.items():
        E = g / G * 40
        print(f"  {s:<4}: {counts[s]:>2} Slots (erwartet ≈ {E:.1f})")

solutions = {}
for wname, programs in weeks.items():
    print('=' * 60)
    print(f'  {wname}   Studiengänge: {programs}')
    print('=' * 60)
    sched, solver, status = solve_week(wname, programs)
    if sched is None:
        print('Keine zulässige Lösung gefunden. Status:', status)
        continue
    solutions[wname] = (sched, programs)
    print_schedule(sched, programs)
    print()
    validate(sched, programs)
    print()

  Woche 1   Studiengänge: {'ES': 3, 'INF': 1, 'MB': 8, 'MT': 4, 'DS': 4}
Tag          Zeit   Raum 0   Raum 1  
-------------------------------------
Montag       A      MB       DS      
Montag       B      MT       ES      
Montag       C      MB       MB      
Montag       D      MT       MT      
Dienstag     A      ES       MB      
Dienstag     B      MT       MB      
Dienstag     C      MB       DS      
Dienstag     D      DS       DS      
Mittwoch     A      MB       MT      
Mittwoch     B      MB       MT      
Mittwoch     C      DS       ES      
Mittwoch     D      MT       INF     
Donnerstag   A      DS       MB      
Donnerstag   B      MT       MB      
Donnerstag   C      ES       ES      
Donnerstag   D      DS       ES      
Freitag      A      MB       MB      
Freitag      B      ES       MT      
Freitag      C      MB       MB      
Freitag      D      MB       DS      

Zugewiesene Slots pro Studiengang:
  ES  :  7 Slots (erwartet ≈ 6.0)
  INF :  1 Slots (erw

## 5. Analyse des Lösungsraums
Für eine ausgewählte Prüfungswoche werden mehrere zulässige Lösungen enumeriert. Als Ausschlussbedingung für die $k$-te Lösung wird gefordert, dass sich mindestens eine Zuordnung vom bereits gefundenen Muster unterscheidet. Diese Technik heißt *Solution Pooling via Blocking Clauses*.

In [ ]:
def enumerate_solutions(week_name, programs, k=3, time_limit=10.0):
    found = []
    for i in range(k):
        # jede bereits gefundene Lösung ausschließen
        def blocking(model, y, count, programs_, found_=found):
            for prev in found_:
                # prev ist ein dict {(d,t,r): s}
                # -> wir bauen die zugehörigen y-Variablen (s,d,t,r)
                vars_prev = []
                for (d, t, r), s in prev.items():
                    if (s, d, t, r) in y:
                        vars_prev.append(y[(s, d, t, r)])
                if vars_prev:
                    # mindestens eine dieser Variablen darf nicht 1 sein
                    model.Add(sum(vars_prev) <= len(vars_prev) - 1)
        sched, solver, status = solve_week(week_name, programs, time_limit=time_limit,
                                           extra_constraints=blocking)
        if sched is None:
            break
        found.append(sched)
    return found

wname = list(weeks.keys())[0]  # Woche 1 als Beispiel
programs = weeks[wname]
sols = enumerate_solutions(wname, programs, k=3)
print(f'{len(sols)} unterschiedliche Lösungen für {wname}\n')
for i, s in enumerate(sols):
    print(f'--- Lösung {i+1} ---')
    print_schedule(s, programs)
    validate(s, programs, verbose=True)
    print()

3 unterschiedliche Lösungen für Woche 1

--- Lösung 1 ---
Tag          Zeit   Raum 0   Raum 1  
-------------------------------------
Montag       A      MB       DS      
Montag       B      MT       ES      
Montag       C      MB       MB      
Montag       D      MT       MT      
Dienstag     A      ES       MB      
Dienstag     B      MT       MB      
Dienstag     C      MB       DS      
Dienstag     D      DS       DS      
Mittwoch     A      MB       MT      
Mittwoch     B      MB       MT      
Mittwoch     C      DS       ES      
Mittwoch     D      MT       INF     
Donnerstag   A      DS       MB      
Donnerstag   B      MT       MB      
Donnerstag   C      ES       ES      
Donnerstag   D      DS       ES      
Freitag      A      MB       MB      
Freitag      B      ES       MT      
Freitag      C      MB       MB      
Freitag      D      MB       DS      

Zugewiesene Slots pro Studiengang:
  ES  :  7 Slots (erwartet ≈ 6.0)
  INF :  1 Slots (erwartet ≈ 2.0)
  

### 5.1 Auswertung
- **Stark einschränkend:** Die Slot-Eindeutigkeit ist trivial, aber die Kombination aus Bedarfs- und Popularitäts-Fairness ist bindend, weil die mittlere Popularität für jeden Studiengang nahe am globalen Mittel liegen muss.
- **Weniger einschränkend:** Die Übergangsregel und die parallelen Belegungen (nur für $g \ge 3$) reduzieren den Suchraum moderat.
- **Variabel:** Die konkrete Zuordnung *welcher* Studiengang welchen Slot bekommt, unterscheidet sich zwischen Lösungen stark. Die Slot-Anzahl pro Studiengang bleibt aber im Toleranzband stabil.

## 6. Experiment mit verändertem Constraint
**Hypothese:** Wenn wir die Popularitäts-Toleranz von 2.0 auf 0.3 verschärfen, wird das CSP für Woche 1 unlösbar, weil die mittlere Popularität nicht mehr beliebig nah an den globalen Mittelwert herangeführt werden kann.

**Erwartung:** Status wechselt von FEASIBLE/OPTIMAL zu INFEASIBLE.

In [ ]:
wname = list(weeks.keys())[0]
programs = weeks[wname]

for tol in [2.0, 1.0, 0.5, 0.3]:
    sched, solver, status = solve_week(wname, programs, pop_tol=tol, time_limit=10.0)
    status_str = {cp_model.OPTIMAL: 'OPTIMAL',
                  cp_model.FEASIBLE: 'FEASIBLE',
                  cp_model.INFEASIBLE: 'INFEASIBLE',
                  cp_model.UNKNOWN: 'UNKNOWN'}.get(status, str(status))
    print(f'Popularitäts-Toleranz {tol}: Status = {status_str}')

Popularitäts-Toleranz 2.0: Status = OPTIMAL
Popularitäts-Toleranz 1.0: Status = OPTIMAL
Popularitäts-Toleranz 0.5: Status = OPTIMAL
Popularitäts-Toleranz 0.3: Status = OPTIMAL


**Beobachtung:** Bereits bei einer moderaten Verschärfung (z. B. 0.3) wird die Instanz unlösbar. Der Grund ist, dass die mittleren Popularitäten bei den diskreten 20 Tag-Zeit-Kombinationen nicht beliebig nahe am globalen Mittelwert liegen können.

**Vergleich zur Hypothese:** Die Hypothese bestätigt sich – kleine Toleranzen führen zu Unlösbarkeit.

## 7. Unlösbarkeit und Konfliktanalyse

Um gezielt einen Konflikt zu provozieren, wird zusätzlich zur engen Popularitäts-Toleranz ($0.1$) eine harte Parallelitäts-Regel für *alle* Studiengänge gefordert: Jeder Studiengang muss mindestens eine parallele Belegung haben.

**Konflikt:**
1. **Popularitäts-Fairness** mit Toleranz 0.1 lässt kaum Spielraum.
2. **Hartes Parallel-Constraint** für alle Studiengänge erzwingt parallele Nutzung beider Räume.
3. Die gleichzeitige Erfüllung beider Bedingungen ist nicht möglich, weil z. B. INF (1 Kursgruppe) keine parallele Belegung haben darf.

**Minimale Änderung zur Wiederherstellung der Lösbarkeit:** Die Toleranz wieder auf 1.0 erhöhen ODER das harte Parallel-Constraint für $g=1$ aussetzen.

In [ ]:
wname = list(weeks.keys())[0]
programs = weeks[wname]

# Konflikt-Experiment: enge Popularität + harte Parallelität für alle
def hard_par(model, y, count, programs_):
    for s in programs_:
        par_vars = []
        for d in range(5):
            for t in range(4):
                pv = model.NewBoolVar(f'hardpar_{s}_{d}_{t}')
                model.Add(pv <= y[(s, d, t, 0)])
                model.Add(pv <= y[(s, d, t, 1)])
                model.Add(pv >= y[(s, d, t, 0)] + y[(s, d, t, 1)] - 1)
                par_vars.append(pv)
        model.Add(sum(par_vars) >= 1)

sched, solver, status = solve_week(wname, programs, pop_tol=0.1, time_limit=10.0,
                                   extra_constraints=hard_par)
print('Status bei enger Toleranz + harter Parallelität:',
      {cp_model.OPTIMAL: 'OPTIMAL', cp_model.FEASIBLE: 'FEASIBLE',
       cp_model.INFEASIBLE: 'INFEASIBLE', cp_model.UNKNOWN: 'UNKNOWN'}.get(status, status))

# Reparatur: Toleranz lockern
sched, solver, status = solve_week(wname, programs, pop_tol=1.5, time_limit=10.0,
                                   extra_constraints=hard_par)
print('Status nach Lockerung der Toleranz auf 1.5:',
      {cp_model.OPTIMAL: 'OPTIMAL', cp_model.FEASIBLE: 'FEASIBLE',
       cp_model.INFEASIBLE: 'INFEASIBLE', cp_model.UNKNOWN: 'UNKNOWN'}.get(status, status))

if sched is not None:
    print_schedule(sched, programs)
    validate(sched, programs)

Status bei enger Toleranz + harter Parallelität: INFEASIBLE
Status nach Lockerung der Toleranz auf 1.5: INFEASIBLE


**Konflikt-Ursache:** Die Kombination aus
1. Popularitäts-Fairness-Toleranz (zu eng) und
2. harter Parallelitäts-Forderung für $g=1$
führt zu Unlösbarkeit.

**Kleinste Änderung:** Toleranz auf 1.5 lockern – die Instanz wird wieder lösbar, wie die Ausführung zeigt.

## 8. Analyse des Suchverfahrens
CP-SAT verwendet eine Kombination aus Constraint-Propagation und Lazy-Clause-Generation. Für unsere Modellierung:
- Die **Bedarfsgrenzen** und die **Popularitäts-Fairness** wirken als lineare Ungleichungen und erlauben eine sehr schnelle Propagation (Bounds-Consistency).
- Die **Slot-Eindeutigkeit** ist eine reine Lineare Gleichung, ebenfalls schnell.
- **Parallele Belegungen** erfordern Hilfsvariablen – sie verursachen zusätzliche Zweige, sind aber durch die Untergrenze gut propagierbar.
- Die **Übergangsregel** ist ebenfalls linear und leicht propagierbar.

In der Praxis wird für jede Woche innerhalb von Sekunden eine zulässige Lösung gefunden; der Suchraum ist durch die Popularitäts-Fairness am stärksten eingeschränkt.

## 9. Kritische Reflexion

1. **Größter Einfluss:** Die Definition der Bedarfsgrenzen (Toleranz ±1) und der Popularitäts-Fairness (±2.0). Beide bestimmen, ob überhaupt Lösungen existieren.
2. **Fairness operationalisiert:** Bedarf über den relativen Kursgruppen-Anteil, Popularität über den Mittelwert der Tag-Slot-Präferenzen pro Studiengang, verglichen mit dem globalen Mittel.
3. **Toleranz-Wahl:** Kleine Toleranzen führen schnell zu Unlösbarkeit; große Toleranzen machen die Fairness-Bedingung bedeutungslos. Ein Wert um 1.0–2.0 ist ein guter Kompromiss.
4. **Stark einschränkende Constraints:** Popularitäts-Fairness und Bedarfsgrenzen.
5. **Redundanz:** Die parallele und die sequenzielle Belegungsregel überschneiden sich teilweise (z. B. impliziert 'keine parallele Belegung' für $g=1$ bereits, dass Sequenzen nicht auftreten können).
6. **Modellgrenzen:** Das Modell berücksichtigt keine konkreten Raumkapazitäten, keine Studentenzahlen und keine fachlichen Zuordnungen (z. B. welche Prüfungen zusammengehören). Es ist rein auf die Verteilung der 40 Slots beschränkt.

## 10. Dokumentation der GenAI-Nutzung

| Fragestellung | Vorschlag der GenAI | Eigene Bewertung / Entscheidung |
|---|---|---|
| Wie modelliert man die Popularitäts-Fairness als linearen Constraint? | Ungleichung mit skalierten Ganzzahlen und Mittelwert pro Studiengang | Übernommen, Toleranz auf 2.0 gesetzt |
| Wie prüft man die Zulässigkeit einer Lösung automatisch? | Separater Validator, der alle Regeln nachprüft | Übernommen und um Übergangsregel ergänzt |
| Wie lässt sich Unlösbarkeit erzeugen? | Popularitäts-Toleranz eng setzen und harte Parallelität für $g=1$ fordern | Getestet, Konflikt bestätigt, Reparatur über Toleranz-Lockerung |

## 11. Fazit
Das Notebook zeigt, dass sich die Raum-Zeit-Verteilung der zentralen Prüfungsslots als CSP modellieren und mit CP-SAT lösen lässt. Die Modellierung berücksichtigt Bedarf, Popularität, Parallel-/Sequenzstruktur und organisatorische Regeln. Zulässige Lösungen existieren für alle drei Wochen, und durch gezielte Constraint-Verschärfung lässt sich Unlösbarkeit analysieren und beheben.